# Predicting Stellar Class

Stellar classification is the process of identifying objects by the spectral patterns in their light. By examining properties such as temperature, composition, and distance, we gain insight into how stars, galaxies, and quasars form and evolve. Effective classification lets us map large surveys, compare objects consistently, and study the structure and history of the universe.

Yao Yan, Walter Reade, Elizabeth Park. Predicting Stellar Class. https://kaggle.com/competitions/playground-series-s6e6, 2026. Kaggle.

## About the data
The data consists of `577347` generated from [observations of space taken by the SDSS (Sloan Digital Sky Survey)](https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17/data). Every observation is described by `10` feature columns and 1 class column which identifies it to be either a star, galaxy or quasar.


| **Field** | **Description** |
| --- | --- |
| **alpha** | Right Ascension angle (J2000 epoch) |
| **delta** | Declination angle (J2000 epoch) |
| **u** | Ultraviolet filter magnitude in the SDSS photometric system |
| **g** | Green filter magnitude |
| **r** | Red filter magnitude |
| **i** | Near‑infrared filter magnitude |
| **z** | Infrared filter magnitude |
| **redshift** | Redshift value based on wavelength stretching |
| **spectral_type** | Spectral classification derived from the object’s spectrum |
| **galaxy_population** | Galaxy population category (e.g., early‑type, late‑type) |
| **class** | Object class label (galaxy, star, or quasar) |

In [2]:
pip install optuna-integration[lightgbm] --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 4.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# imports
import pandas as pd
import numpy as np
import optuna
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna.integration import LightGBMPruningCallback, XGBoostPruningCallback

try:
    from IPython.core.magic import register_cell_magic
    @register_cell_magic
    def skip(line, cell): return
except:
    pass

In [4]:
# load data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv', index_col="id")
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv', index_col="id")
submission_sample = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv')

In [5]:
def downcasting(data: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    mem_before = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {mem_before:.2f} MB")
    
    for col in data.select_dtypes(include=["number"]).columns:
        if pd.api.types.is_integer_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="integer")
        elif pd.api.types.is_float_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="float")
    
    mem_after = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {mem_after:.2f} MB")
        print(f"Decreased by {(100 * (mem_before - mem_after) / mem_before):.1f}%\n")
    
    return data

print("Train train:")
train = downcasting(train)
print("Test train:")
test = downcasting(test)

Train train:
Memory usage of dataframe is 52.86 MB
Memory usage after optimization is: 35.24 MB
Decreased by 33.3%

Test train:
Memory usage of dataframe is 20.77 MB
Memory usage after optimization is: 13.21 MB
Decreased by 36.4%



## EDA 
To understand how each feature relates to the target class, I used Cramér’s V for the categorical features and Mutual Information (MI) for the numeric features. 

The results show two very strong categorical predictors (`galaxy_population`, `spectral_type`) and one very strong numeric predictor (`redshift`, while the photometric magnitudes (`u`, `g`, `r`, `i`, `z`) provide moderate signal and sky coordinates (`alpha`, `delta`) contribute weak signal. This pattern is typical of SDSS‑style astronomy datasets.

These findings guide the feature engineering:

- Color indices and magnitude ratios strengthen the moderate photometric features by capturing nonlinear color–color relationships.
- Sin/cos encodings for alpha and delta avoid angular discontinuities and extract positional structure.
- Log‑scaled and zero‑flag variants of redshift capture both continuous and discrete behavior.

In [18]:
df = train.copy()
target = "class"

num_cols = df.select_dtypes(include=["float32", "float64", "int32", "int64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

num_cols = [c for c in num_cols if c != target]
cat_cols = [c for c in cat_cols if c != target]

# mi
mi = mutual_info_classif(df[num_cols], df[target], discrete_features=False)
mi_series = pd.Series(mi, index=num_cols).sort_values(ascending=False)

# cramers
cramers = {}

for col in cat_cols:
    cramers[col] = cramers_v(df[col], df[target])

cramers = pd.Series(cramers).sort_values(ascending=False)

print("=== Feature Importance Summary ===")
print("Categorical (Cramer's V):")
print(cramers)

print("\nNumeric (Mutual Information):")
print(mi_series)

=== Feature Importance Summary ===
Categorical (Cramer's V):
galaxy_population    0.593491
spectral_type        0.524792
dtype: float64

Numeric (Mutual Information):
redshift    0.514972
z           0.211576
u           0.178893
g           0.174517
i           0.158208
alpha       0.138647
delta       0.130072
r           0.097908
dtype: float64


In [17]:
target = "class"
X = train.drop(target, axis=1)
y = train[target].astype("category").cat.codes

## Feature Engineering

- Transform raw photometric measurements into color indices and magnitude ratios to capture nonlinear spectral and temperature differences between object types.
- Encode sky coordinates using sin/cos transformations to handle angular circularity and preserve positional structure.
- Add log‑scaled and zero‑flag variants of redshift to separate stellar objects from extragalactic ones and model both continuous and discrete behavior.

In [ ]:
def feature_engineer(df):
    df = df.copy()
    
    # -----------------------------
    # 1. Color indices
    # -----------------------------
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    # -----------------------------
    # 2. Magnitude ratios
    # -----------------------------
    df["u_over_g"] = df["u"] / df["g"]
    df["g_over_r"] = df["g"] / df["r"]
    df["r_over_i"] = df["r"] / df["i"]

    # -----------------------------
    # 3. Angular encodings
    # -----------------------------
    df["sin_alpha"] = np.sin(np.radians(df["alpha"]))
    df["cos_alpha"] = np.cos(np.radians(df["alpha"]))
    df["sin_delta"] = np.sin(np.radians(df["delta"]))
    df["cos_delta"] = np.cos(np.radians(df["delta"]))

    # -----------------------------
    # 4. Redshift features
    # -----------------------------
    df["log_redshift"] = np.log1p(df["redshift"])
    df["is_zero_redshift"] = (df["redshift"] == 0).astype(int)

    # -----------------------------
    # 5. Encode categorical features
    # -----------------------------
    df["spectral_type"] = df["spectral_type"].astype("category").cat.codes
    df["galaxy_population"] = df["galaxy_population"].astype("category").cat.codes

    return df

X = feature_engineer(X)
test = feature_engineer(test)

## Evaluation Metric: Balanced Accuracy

Balanced accuracy measures how well a classifier performs across all classes by giving each class equal weight, regardless of class imbalance. It is defined as the average recall across all \(K\) classes:

$\text{Balanced Accuracy} = \frac{1}{K} \sum_{i=1}^{K} \text{Recall}_i$

where

$\text{Recall}_i = \frac{\text{TP}_i}{\text{TP}_i + \text{FN}_i}.$

This metric is more effective than standard accuracy for imbalanced datasets because it prevents majority classes from dominating the score and ensures that minority classes contribute equally to the evaluation.


## Modeling
1. Selected tree‑based models because they are well‑suited to structured tabular data and can naturally capture nonlinear relationships present in the engineered astronomical features. Methods such as Random Forest, XGBoost, and LightGBM handle mixed feature types, are robust to outliers, and perform strongly without requiring feature scaling.

2. Applied hyperparameter tuning to optimize model complexity, regularization, and sampling parameters, ensuring that each model generalized well under the balanced‑accuracy metric.

3. Combined the tuned models using a weighted soft‑voting ensemble, allowing models with higher validation balanced accuracy to contribute more strongly to the final prediction and improving overall robustness.

In [ ]:
# ---------------------------------------------------------
# Internal split for tuning
# ---------------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
%%skip
# ---------------------------------------------------------
# 1. Random Forest
# ---------------------------------------------------------
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "n_jobs": -1,
        "random_state": 42
    }

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid, preds)

    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=20)
best_rf_params = study_rf.best_params

In [ ]:
%%skip
# ---------------------------------------------------------
# 2. XGBoost
# ---------------------------------------------------------
def objective_xgb(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 3),
        "tree_method": "hist"
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid, preds)

    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=20)
best_xgb_params = study_xgb.best_params

In [ ]:
%%skip
# ---------------------------------------------------------
# 3. LightGBM
# ---------------------------------------------------------
def objective_lgb(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "num_leaves": trial.suggest_int("num_leaves", 20, 120),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "verbosity": -1
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid, preds)

    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=20)
best_lgb_params = study_lgb.best_params

In [ ]:
best_rf_params = {'n_estimators': 215,
                 'max_depth': 12,
                 'min_samples_split': 10,
                 'min_samples_leaf': 1,
                 'max_features': 'sqrt'}
best_xgb_params = {'n_estimators': 350,
                 'learning_rate': 0.14906285047761964,
                 'max_depth': 7,
                 'subsample': 0.6170554759821828,
                 'colsample_bytree': 0.9891311421595581,
                 'gamma': 0.6568960322808137}
best_lgb_params = {'n_estimators': 452,
                 'learning_rate': 0.0987875952772275,
                 'max_depth': 10,
                 'num_leaves': 103,
                 'subsample': 0.9847557769044509,
                 'colsample_bytree': 0.6220038063150422}

In [ ]:
# ---------------------------------------------------------
# 4. Retrain all tuned models on full X
# ---------------------------------------------------------
rf_final  = RandomForestClassifier(**best_rf_params, n_jobs=-1, random_state=42)
xgb_final = xgb.XGBClassifier(**best_xgb_params)
lgb_final = lgb.LGBMClassifier(**best_lgb_params)

rf_final.fit(X, y)
xgb_final.fit(X, y)
lgb_final.fit(X, y)

In [ ]:
%%skip
# ---------------------------------------------------------
# 5. Tune blending weights with Optuna
# ---------------------------------------------------------
def objective_blend(trial):
    w_rf  = trial.suggest_float("w_rf",  0.0, 1.0)
    w_xgb = trial.suggest_float("w_xgb", 0.0, 1.0)
    w_lgb = trial.suggest_float("w_lgb", 0.0, 1.0)

    total = w_rf + w_xgb + w_lgb + 1e-9

    rf_p  = rf_final.predict_proba(X_valid)
    xgb_p = xgb_final.predict_proba(X_valid)
    lgb_p = lgb_final.predict_proba(X_valid)

    blend_p = (w_rf*rf_p + w_xgb*xgb_p + w_lgb*lgb_p) / total
    blend_pred = np.argmax(blend_p, axis=1)

    return balanced_accuracy_score(y_valid, blend_pred)

study_blend = optuna.create_study(direction="maximize")
study_blend.optimize(objective_blend, n_trials=30)
best_weights = study_blend.best_params

print("Best blending weights:", best_weights)

In [ ]:
best_weights = {'w_rf': 0.001619911443186843,
             'w_xgb': 0.8479234839621768,
             'w_lgb': 0.9784922782231875}

## Prediction

In [ ]:
# ---------------------------------------------------------
# 6. Final blended predictions for test
# ---------------------------------------------------------
w_rf  = best_weights["w_rf"]
w_xgb = best_weights["w_xgb"]
w_lgb = best_weights["w_lgb"]
total = w_rf + w_xgb + w_lgb

rf_p  = rf_final.predict_proba(test)
xgb_p = xgb_final.predict_proba(test)
lgb_p = lgb_final.predict_proba(test)

blend_p = (w_rf*rf_p + w_xgb*xgb_p + w_lgb*lgb_p) / total
final_pred = np.argmax(blend_p, axis=1)
categories = train[target].astype('category').cat.categories
final_labels = categories[final_pred]

In [ ]:
submission = pd.DataFrame({
    'id': test.index, 
    'class': final_labels
})
submission.to_csv('submission.csv', index=False)

In [ ]:
submission